# 02 - El pipeline en acción, etapa por etapa

Este notebook toma contratos reales y muestra cómo se transforma el texto en cada etapa del
pipeline de SECOPEDIA. Sus salidas son las que se citan en el documento final.

Requisitos: haber ejecutado `descargar_datos.py`, `preparar_datos.py`, `construir_lugares.py`,
`preprocesar.py` y `entrenar_linea_base.py`.

In [1]:
import json

import joblib
import pandas as pd

from secopedia.configuracion import RAIZ_PROYECTO, cargar_config, ruta
from secopedia.etiquetas import extraer_segmento
from secopedia.limpieza import clave_texto
from secopedia.normalizacion import Normalizador
from secopedia.representacion import terminos_principales
from secopedia.segmentos import nombre_segmento
from secopedia.tokenizacion import cargar_spacy, lematizar

pd.set_option("display.max_colwidth", 200)
config = cargar_config()
normalizador = Normalizador.desde_archivo(ruta(config["rutas"]["lugares"]))
nlp = cargar_spacy()
modelo = joblib.load(RAIZ_PROYECTO / config["rutas"]["modelos"] / "regresion_logistica.joblib")
vectorizador = modelo.named_steps["tfidf"]

## Contratos de ejemplo

Se usan contratos de la tabla de muestra de la Entrega 1 (texto y código tal como están en SECOP II).

In [2]:
ejemplos = pd.DataFrame([
    ("COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023", "V1.50201700"),
    ("ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA", "V1.84131500"),
    ("Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander", "V1.15101506"),
    ("Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia", "V1.42121500"),
    ("Adquisición de mobiliario para las sedes de la Delegación Departamental de la Registraduría Nacional del Estado Civil en el departamento de Magdalena.", "V1.72153606"),
], columns=["objeto_del_contrato", "codigo_de_categoria_principal"])
ejemplos

,objeto_del_contrato,codigo_de_categoria_principal
0,COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023,V1.50201700
1,ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA,V1.84131500
2,Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander,V1.15101506
3,Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia,V1.42121500
4,Adquisición de mobiliario para las sedes de la Delegación Departamental de la Registraduría Nacional del Estado Civil en el departamento de Magdalena.,V1.72153606


## Etapas 2 y 3: clave de deduplicación y etiqueta

La clave de texto (minúsculas, sin tildes ni signos) detecta duplicados. La etiqueta es el
segmento: los 2 primeros dígitos del código UNSPSC.

In [3]:
clases = json.loads((ruta(config["rutas"]["metricas"]) / "clases.json").read_text(encoding="utf-8"))
etapa_2_3 = pd.DataFrame({
    "clave_texto": ejemplos["objeto_del_contrato"].map(clave_texto),
    "codigo": ejemplos["codigo_de_categoria_principal"],
    "segmento": ejemplos["codigo_de_categoria_principal"].map(extraer_segmento),
})
etapa_2_3["etiqueta"] = etapa_2_3["segmento"].where(etapa_2_3["segmento"].isin(clases), "otros")
etapa_2_3["nombre"] = etapa_2_3["etiqueta"].map(nombre_segmento)
etapa_2_3

,clave_texto,codigo,segmento,etiqueta,nombre
0,compra de productos de cafeteria para la entidad municipio de yumbo vigencia 2023,V1.50201700,50,50,"Alimentos, bebidas y tabaco"
1,adquisicion de la poliza de manejo global para la personeria municipal de sotaquira boyaca,V1.84131500,84,84,Servicios financieros y de seguros
2,contratar el suministro de combustible lubricantes llantas y baterias para los vehiculos y el aula movil del centro agroturistico regional santander,V1.15101506,15,15,"Combustibles, aditivos para combustibles, lubricantes y anticorrosivos"
3,adquisicion de implementos insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de caldas antioquia,V1.42121500,42,42,"Equipo médico, accesorios y suministros"
4,adquisicion de mobiliario para las sedes de la delegacion departamental de la registraduria nacional del estado civil en el departamento de magdalena,V1.72153606,72,72,"Servicios de edificación, construcción y mantenimiento"


## Etapa 4: normalización

Minúsculas, sin tildes, separadores a espacios, años a `<AÑO>`, números a `<NUM>` y
municipios o departamentos a `<LUGAR>`.

In [4]:
ejemplos["normalizado"] = ejemplos["objeto_del_contrato"].map(normalizador)
for original, normalizado in zip(ejemplos["objeto_del_contrato"], ejemplos["normalizado"]):
    print(f"ORIGINAL:    {original}\nNORMALIZADO: {normalizado}\n")

ORIGINAL:    COMPRA DE PRODUCTOS DE CAFETERIA PARA LA ENTIDAD MUNICIPIO DE YUMBO VIGENCIA 2023
NORMALIZADO: compra de productos de cafeteria para la entidad municipio de <LUGAR> vigencia <AÑO>

ORIGINAL:    ADQUISICION DE LA POLIZA DE MANEJO GLOBAL PARA LA PERSONERIA MUNICIPAL DE SOTAQUIRA BOYACA
NORMALIZADO: adquisicion de la poliza de manejo global para la personeria municipal de <LUGAR>

ORIGINAL:    Contratar el suministro de combustible; lubricantes; llantas y baterías para los vehículos y el aula móvil del Centro Agroturístico Regional Santander
NORMALIZADO: contratar el suministro de combustible lubricantes llantas y baterias para los vehiculos y el aula movil del centro agroturistico regional <LUGAR>

ORIGINAL:    Adquisición de implementos; insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio de Caldas; Antioquia
NORMALIZADO: adquisicion de implementos insumos y medicamentos para prestar los servicios de medicina veterinaria en el municipio

## Etapa 5: tokenización, stopwords y lematización

spaCy (`es_core_news_sm`) separa los tokens, quita stopwords del español y del dominio
(por ejemplo "adquisicion", "contratar", "municipio") y deja el lema de cada palabra.

In [5]:
ejemplos["lemas"] = list(lematizar(ejemplos["normalizado"], nlp))
doc = nlp(ejemplos["normalizado"].iat[0])
print("Tokens de spaCy del primer ejemplo:")
print(pd.DataFrame([(t.text, t.lemma_, t.is_stop) for t in doc], columns=["token", "lema", "stopword"]).T.to_string(header=False))
print()
for normalizado, lemas in zip(ejemplos["normalizado"], ejemplos["lemas"]):
    print(f"{normalizado}\n  -> {lemas}\n")

Tokens de spaCy del primer ejemplo:
token     compra    de  productos    de  cafeteria  para    la  entidad  municipio    de  <LUGAR>  vigencia  <AÑO>
lema      compra    de   producto    de  cafeteria  para    el  entidad  municipio    de  <lugar>  vigencia  <año>
stopword   False  True      False  True      False  True  True    False      False  True    False     False  False

compra de productos de cafeteria para la entidad municipio de <LUGAR> vigencia <AÑO>
  -> ['producto', 'cafeteria']

adquisicion de la poliza de manejo global para la personeria municipal de <LUGAR>
  -> ['poliza', 'manejo', 'global', 'personeria']

contratar el suministro de combustible lubricantes llantas y baterias para los vehiculos y el aula movil del centro agroturistico regional <LUGAR>
  -> ['suministro', 'combustible', 'lubricant', 'llanta', 'bateria', 'vehiculo', 'aula', 'movil', 'centro', 'agroturistico', 'regional']

adquisicion de implementos insumos y medicamentos para prestar los servicios de med

## Etapa 6: representación TF-IDF

Cada contrato se convierte en un vector disperso de unigramas y bigramas de lemas. Se muestran
los términos con más peso en cada ejemplo.

In [6]:
print(f"Tamaño del vocabulario TF-IDF: {len(vectorizador.vocabulary_):,} términos")
for lemas in ejemplos["lemas"]:
    texto = " ".join(lemas)
    print(f"{texto}\n  -> {terminos_principales(vectorizador, texto)}\n")

Tamaño del vocabulario TF-IDF: 85,756 términos
producto cafeteria
  -> [('producto cafeteria', 0.709), ('cafeteria', 0.556), ('producto', 0.434)]

poliza manejo global personeria
  -> [('global personeria', 0.573), ('poliza manejo', 0.398), ('manejo global', 0.371), ('global', 0.354), ('personeria', 0.327)]

suministro combustible lubricant llanta bateria vehiculo aula movil centro agroturistico regional
  -> [('vehiculo aula', 0.313), ('lubricant llanta', 0.295), ('agroturistico regional', 0.291), ('bateria vehiculo', 0.291), ('centro agroturistico', 0.278)]

implemento insumos medicamento medicina veterinario
  -> [('medicina veterinario', 0.469), ('implemento insumos', 0.457), ('insumos medicamento', 0.451), ('veterinario', 0.311), ('medicina', 0.285)]

mobiliario sed delegacion registraduria nacional civil
  -> [('mobiliario sed', 0.44), ('delegacion registraduria', 0.426), ('nacional civil', 0.349), ('registraduria nacional', 0.349), ('delegacion', 0.308)]



## Etapas 7 y 9: predicción (top 3)

Probabilidades de la regresión logística para cada ejemplo. El último contrato está registrado
como 72 (construcción) pero su texto habla de mobiliario (56): es un caso de etiqueta ruidosa
que el modo *Auditar* detectaría.

In [7]:
probabilidades = modelo.predict_proba(ejemplos["lemas"].map(" ".join))
for i, fila in ejemplos.iterrows():
    orden = probabilidades[i].argsort()[::-1][:3]
    top3 = ", ".join(f"{modelo.classes_[j]} ({probabilidades[i][j]:.2f})" for j in orden)
    print(f"Registrado: {etapa_2_3['etiqueta'].iat[i]:>5} | Top 3: {top3}")

Registrado:    50 | Top 3: 50 (0.99), 90 (0.00), 47 (0.00)
Registrado:    84 | Top 3: 84 (1.00), 93 (0.00), otros (0.00)
Registrado:    15 | Top 3: 15 (0.68), 78 (0.20), 25 (0.11)
Registrado:    42 | Top 3: 42 (0.41), 70 (0.40), otros (0.04)
Registrado:    72 | Top 3: 56 (0.92), 43 (0.02), 44 (0.01)
